# Toy Gap Convergence Debug Notebook

This notebook is a guided, heavily documented version of the `toy_gap_convergence_debug.py` parity harness.

The goal is to make the debugging workflow easier to follow: you can inspect a tiny BBOB problem, compare the `evosax` and `malthusjax` paths, and replay captured parity traces step by step.

The notebook is intentionally verbose. It is designed to explain what each block is doing, why the block exists, and how it relates to parity debugging.

## What This Notebook Covers

1. Set up a tiny BBOB problem and a configurable debug run.
2. Build shared helpers for reporting gaps, populations, and operator state.
3. Run the `evosax` and `malthusjax` backends on the same problem.
4. Replay captured crossover and selection parity traces.
5. Compare repeated trials when you want a broader statistical view.

The notebook keeps the core logic from the script, but it presents it in a more explanatory, notebook-friendly order.

In [7]:
from __future__ import annotations

import dataclasses
from pathlib import Path

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import optax

from malthusjax.composer.evosax_adapter import build_evosax_engine
from malthusjax.core.fitness.bbob_evaluator import BBOBConfig, BBOBEvaluator
from malthusjax.core.genome.real_genome import RealGenomeConfig, RealPopulation
from malthusjax.engine.genetic_fastengine import GeneticEngine, GeneticEngineParams
from malthusjax.engine.resource_mapper import get_resource_summary, get_step_dimension_flow
from malthusjax.operators.crossover.evosax_crossover import EvosaxUniformCrossoverWrapper
from malthusjax.operators.mutation.evosax_mutation import EvosaxGaussianWrapper
from malthusjax.operators.selection.elite_pool import ElitePoolSelection

print(f'JAX version: {jax.__version__}')
print(f'Default backend: {jax.default_backend()}')

JAX version: 0.10.0
Default backend: cpu


## Experiment Configuration

The original script uses command-line arguments. In a notebook, it is clearer to keep the settings in one editable cell so you can rerun experiments without restarting Python.

The important pieces for parity are:
- `function`: which BBOB objective we compare.
- `dimensions`: the genome dimensionality.
- `pop_size`: the population size.
- `elite_k`: how many elites are preserved by the MalthusJAX selection path.
- `crossover_rate` and `mutation_strength`: operator parameters that need to be aligned between systems.

In [8]:
# Notebook configuration
backend = 'both'  # options: 'evosax', 'malthusjax', 'both'
function = 'rosenbrock'
dimensions = 3
pop_size = 12
generations = 20
seed = 0
elite_k = None  # if None, default to max(1, pop_size // 2)
crossover_rate = 0.1
mutation_strength = 0.0
show_resource_summary = False
show_dimension_flow = False
plot = False
output = Path('results/toy_gap_convergence_debug.png')

compare_trials = 100
capture_trials = 100
capture_top_k = 0
param_sweep = False
replay_crossover_parity_mode = False
replay_selection_parity_mode = False

print('Configured notebook settings:')
print({
    'backend': backend,
    'function': function,
    'dimensions': dimensions,
    'pop_size': pop_size,
    'generations': generations,
    'seed': seed,
    'elite_k': elite_k,
    'crossover_rate': crossover_rate,
    'mutation_strength': mutation_strength,
})

Configured notebook settings:
{'backend': 'both', 'function': 'rosenbrock', 'dimensions': 3, 'pop_size': 12, 'generations': 20, 'seed': 0, 'elite_k': None, 'crossover_rate': 0.1, 'mutation_strength': 0.0}


## Helper Functions

These helpers keep the rest of the notebook readable. They do not change the experiment logic; they only format outputs, summarize gaps, and print the live state of populations and operators.

The two most important reporting conventions are:
- **gap history**: the absolute distance from the known optimum.
- **best-so-far**: the running minimum of the gap history, which makes convergence easier to read.

In [9]:
def _best_so_far(gap_history: list[float]) -> list[float]:
    return np.minimum.accumulate(gap_history).tolist()

def _print_report(label: str, gap_history: list[float], best_so_far: list[float]) -> None:
    print(f'\n== {label} ==')
    print('generation, gap_to_optimum, best_so_far')
    for generation, gap, best in zip(range(0, len(gap_history)), gap_history, best_so_far):
        print(f'{generation:>3d}, {gap:>12.6f}, {best:>12.6f}')
    print()
    print(f'initial gap: {gap_history[0]:.6f}')
    print(f'final gap:   {gap_history[-1]:.6f}')
    print(f'best-so-far:  {best_so_far[-1]:.6f}')

def _print_population_snapshot(label: str, population: RealPopulation) -> None:
    fitness = np.asarray(population.fitness)
    genes = np.asarray(population.genes)
    print(f'{label}: size={len(population)}, gene_shape={genes.shape[1:]}')
    print(
        f'{label}: fitness min/mean/max = '
        f'{fitness.min():.6f} / {fitness.mean():.6f} / {fitness.max():.6f}'
    )
    print(f'{label}: first_fitness={fitness[: min(3, len(fitness))].tolist()}')

def _format_value(value: object) -> str:
    if isinstance(value, (int, float, bool, str)):
        return repr(value)
    if isinstance(value, np.ndarray):
        if value.size <= 8:
            return repr(value.tolist())
        return f'array(shape={value.shape}, dtype={value.dtype})'
    if hasattr(value, 'shape') and hasattr(value, 'dtype'):
        return f'array(shape={tuple(value.shape)}, dtype={value.dtype})'
    return repr(value)

def _print_operator_box(title: str, operator: object, result: object) -> None:
    fields = []
    if dataclasses.is_dataclass(operator):
        for field in dataclasses.fields(operator):
            fields.append((field.name, getattr(operator, field.name)))

    type_name = type(operator).__name__
    module_name = type(operator).__module__
    lines = [
        f"┌─ {title} ─{'─' * max(0, 44 - len(title))}┐",
        f'│ type: {module_name}.{type_name}',
        f'│ result: {_format_value(result)}',
        '│ parameters:',
    ]
    for name, value in fields:
        lines.append(f'│   {name}: {_format_value(value)}')
    lines.append(f"└{'─' * 58}┘")
    print('\n'.join(lines))

def _mean_euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape != b.shape:
        raise ValueError(f'shape mismatch for distance computation: {a.shape} vs {b.shape}')
    if a.ndim == 1:
        return float(np.linalg.norm(a - b))
    return float(np.mean(np.linalg.norm(a - b, axis=-1)))

## Shared Parity Setup

The notebook uses a single initial population for both systems. This is one of the most important parity controls because it removes one entire source of random variation.

We also centralize the shared strategy parameters here. That keeps the `evosax` and `malthusjax` paths aligned when we compare operator behavior.

In [10]:
def _shared_initial_population() -> jax.Array:
    evaluator = BBOBEvaluator.create(
        BBOBConfig(
            fn_name=function,
            num_dims=dimensions,
            seed=seed,
            maximize=False,
        )
    )
    key = jr.PRNGKey(seed)
    initial_keys = jr.split(key, pop_size)
    return jax.vmap(evaluator.evosax_problem.sample)(initial_keys)

def _shared_strategy_params(elite_k_value: int) -> dict[str, object]:
    return {
        'crossover_rate': crossover_rate,
        'elite_ratio': float(elite_k_value) / float(pop_size),
        'std_schedule': optax.constant_schedule(mutation_strength),
    }

def _resolved_elite_k() -> int:
    return elite_k if elite_k is not None else max(1, pop_size // 2)

elite_k_value = _resolved_elite_k()
print(f'resolved elite_k = {elite_k_value}')
print(f'shared elite_ratio = {elite_k_value / pop_size:.3f}')

resolved elite_k = 6
shared elite_ratio = 0.500


## `evosax` Backend

This cell sets up the reference backend and runs a single toy experiment. The notebook prints the gap-to-optimum after each generation so you can visually inspect convergence rather than only looking at a final score.

The key idea is simple: sample one shared initial population, initialize the backend, run it, and record the absolute gap to the optimum after each step.

In [11]:
def run_evosax() -> tuple[list[float], list[float]]:
    evaluator = BBOBEvaluator.create(
        BBOBConfig(
            fn_name=function,
            num_dims=dimensions,
            seed=seed,
            maximize=False,
        )
    )

    adapter = build_evosax_engine(
        strategy_name='SimpleGA',
        evaluator=evaluator,
        pop_size=pop_size,
        generations=generations,
        bounds=(-5.0, 5.0),
        maximize=False,
        seed=seed,
        strategy_params=_shared_strategy_params(elite_k_value),
        initial_population=_shared_initial_population(),
    )

    population_init = _shared_initial_population()
    key_eval = jr.PRNGKey(seed + 1)
    fitness_init, _, _ = evaluator.evosax_problem.eval(
        key_eval, population_init, evaluator.problem_state
    )

    # The next block is intentionally diagnostic: it peeks inside the backend
    # so that we can compare the sampled offspring and the internal parameter
    # objects against the MalthusJAX path.
    strategy = adapter.strategy
    params = adapter.params
    rng = jr.PRNGKey(seed + 1)
    rng, key_init, key_ask = jr.split(rng, 3)
    state = strategy.init(key_init, population_init, fitness_init, params)
    offspring_pop, state_after = strategy.ask(key_ask, state, params)

    ev_params = {}
    try:
        ev_params.update(getattr(params, '__dict__', {}))
    except Exception:
        pass

    strat_attrs = {k: v for k, v in strategy.__dict__.items() if not k.startswith('_')}

    _print_operator_box(
        'evosax strategy (ask) output',
        strategy,
        {
            'offspring_shape': np.asarray(offspring_pop).shape,
            'first_offspring': np.asarray(offspring_pop)[: min(3, np.asarray(offspring_pop).shape[0])].tolist(),
        },
    )

    _print_operator_box(
        'evosax strategy params',
        params,
        {'params': ev_params, 'strategy_attrs': {k: _format_value(v) for k, v in strat_attrs.items()}},
    )

    initial_best_idx = int(np.argmin(np.asarray(fitness_init)))
    initial_gap = abs(float(fitness_init[initial_best_idx]) - float(evaluator.f_opt))

    result = adapter.run_once(jr.PRNGKey(seed), compile=False)
    gap_history = [initial_gap] + [
        abs(float(row['best_fitness']) - float(evaluator.f_opt)) for row in result['history']
    ]
    return gap_history, _best_so_far(gap_history)

evosax_gap_history, evosax_best_so_far = run_evosax()
_print_report('evosax', evosax_gap_history, evosax_best_so_far)

┌─ evosax strategy (ask) output ─────────────────┐
│ type: evosax.algorithms.population_based.simple_ga.SimpleGA
│ result: {'offspring_shape': (12, 3), 'first_offspring': [[0.2873826026916504, 2.290053367614746, -4.021965026855469], [-0.7644712924957275, -1.5434801578521729, 4.293442726135254], [-2.40814208984375, -4.366184234619141, 2.3858094215393066]]}
│ parameters:
└──────────────────────────────────────────────────────────┘
┌─ evosax strategy params ───────────────────────┐
│ type: evosax.algorithms.population_based.simple_ga.Params
│ result: {'params': {'crossover_rate': 0.1}, 'strategy_attrs': {'population_size': '12', 'solution': 'array(shape=(3,), dtype=float32)', 'metrics_fn': '<function metrics_fn at 0x11991d080>', 'solution_flat': 'array(shape=(3,), dtype=float32)', 'num_dims': '3', 'fitness_shaping_fn': '<function identity_fitness_shaping_fn at 0x1198f3a60>', 'elite_ratio': '0.5', 'max_num_dims_sq': 'array(shape=(), dtype=float32)', 'std_schedule': '<function constant_sche

## `malthusjax` Backend

This is the native engine path. The notebook walks through the same overall problem setup, but it also prints resource-flow and operator-level information that is specific to `MalthusJAX`.

This is the main place where you can see how the internal population, selection, crossover, and mutation stages are wired together.

In [12]:
def run_malthusjax() -> tuple[list[float], list[float]]:
    genome_config = RealGenomeConfig(shape=(dimensions,), bounds=(-5.0, 5.0))
    engine_params = GeneticEngineParams(
        pop_size=pop_size,
        elitism=0,
        num_generations=generations,
        forward_presplit_keys=True,
    )
    evaluator = BBOBEvaluator.create(
        BBOBConfig(
            fn_name=function,
            num_dims=dimensions,
            seed=seed,
            maximize=False,
        )
    )

    engine = GeneticEngine(
        engine_params=engine_params,
        genome_config=genome_config,
        evaluator=evaluator,
        selection=ElitePoolSelection(num_selections=pop_size, elite_k=elite_k_value),
        crossover=EvosaxUniformCrossoverWrapper(
            num_offspring=1,
            crossover_rate=crossover_rate,
        ),
        mutation=EvosaxGaussianWrapper(
            num_offspring=1,
            mutation_strength=mutation_strength,
        ),
        enable_progress_bar=False,
    )

    print(
        '\n== malthusjax debug config ==\n'
        f'function={function}, dims={dimensions}, pop_size={pop_size}, '
        f'generations={generations}, elite_k={elite_k_value}, '
        f'crossover_rate={crossover_rate}, mutation_strength={mutation_strength}'
    )

    shared_population = _shared_initial_population()
    initial_population = RealPopulation.from_array(shared_population, genome_config, axis=0)
    evaluated_population = evaluator.evaluate_population(initial_population)
    initial_best_idx = int(np.argmin(np.asarray(evaluated_population.fitness)))
    _print_population_snapshot('initial population', evaluated_population)

    init_state = engine.init_state(jr.PRNGKey(seed))

    if show_resource_summary:
        print(get_resource_summary(init_state.resource_map))

    if show_dimension_flow:
        print(
            get_step_dimension_flow(
                init_state.resource_map,
                elitism=engine_params.elitism,
                pop_symbol='n',
                genome_symbol='d',
                genome_width=dimensions,
            )
        )

    state = init_state.replace(
        population=evaluated_population,
        best_genome=evaluated_population.genes[initial_best_idx],
        best_fitness=evaluated_population.fitness[initial_best_idx],
    )
    best_history = [float(state.best_fitness)]

    # The following diagnostic block makes the operator flow visible.
    # It mirrors the engine's live selection/crossover/mutation path so that
    # parity issues can be located at the exact stage where they appear.
    k_sel, k_cross, k_mut, k_next = engine._allocate_entropy(state)
    elites_genes, parent_indices = engine._selection_phase(
        k_sel, state.population, state.operators, engine.engine_params
    )

    rmap = state.resource_map
    num_pairs = rmap.crossover.input_count // 2
    p1_idx = parent_indices[:num_pairs]
    p2_idx = parent_indices[num_pairs : num_pairs * 2]
    p1_genes = jax.tree_util.tree_map(lambda x: x[p1_idx], state.population.genes)
    p2_genes = jax.tree_util.tree_map(lambda x: x[p2_idx], state.population.genes)
    dummy_fitness = jnp.zeros(num_pairs)
    p1_pop = state.population.spawn_offspring(p1_genes, fitness=dummy_fitness)
    p2_pop = state.population.spawn_offspring(p2_genes, fitness=dummy_fitness)

    crossover_offspring = state.operators.crossover(
        k_cross, p1_pop, p2_pop, engine.genome_config, generation=state.generation
    )
    mutated_offspring = state.operators.mutation(
        k_mut, crossover_offspring, engine.genome_config, generation=state.generation
    )

    _print_operator_box(
        'live crossover operator',
        state.operators.crossover,
        {
            'offspring_shape': np.asarray(jax.tree_util.tree_leaves(crossover_offspring.genes)[0]).shape,
            'first_offspring': np.asarray(jax.tree_util.tree_leaves(crossover_offspring.genes)[0])[: min(3, np.asarray(jax.tree_util.tree_leaves(crossover_offspring.genes)[0]).shape[0])].tolist(),
        },
    )

    _print_operator_box(
        'live mutation operator',
        state.operators.mutation,
        {
            'offspring_shape': np.asarray(jax.tree_util.tree_leaves(mutated_offspring.genes)[0]).shape,
            'first_offspring': np.asarray(jax.tree_util.tree_leaves(mutated_offspring.genes)[0])[: min(3, np.asarray(jax.tree_util.tree_leaves(mutated_offspring.genes)[0]).shape[0])].tolist(),
        },
    )

    final_state, history = engine.debug_run(state)
    _ = final_state
    best_history.extend(float(item.best_fitness) for item in history)

    _print_population_snapshot('final population', final_state.population)

    live_selection = final_state.operators.selection
    selection_parent_idx, selection_elite_idx = live_selection(state.rng_key, state.population)
    _print_operator_box(
        'live selection operator',
        live_selection,
        {
            'parent_idx': np.asarray(selection_parent_idx).tolist(),
            'elite_idx': np.asarray(selection_elite_idx).tolist(),
        },
    )

    gap_history = [abs(value - float(evaluator.f_opt)) for value in best_history]
    return gap_history, _best_so_far(gap_history)

malthusjax_gap_history, malthusjax_best_so_far = run_malthusjax()
_print_report('malthusjax', malthusjax_gap_history, malthusjax_best_so_far)


== malthusjax debug config ==
function=rosenbrock, dims=3, pop_size=12, generations=20, elite_k=6, crossover_rate=0.1, mutation_strength=0.0
initial population: size=12, gene_shape=(3,)
initial population: fitness min/mean/max = 320.682007 / 95073.929688 / 290727.093750
initial population: first_fitness=[247403.90625, 79159.609375, 240966.484375]


/var/folders/n8/08b2nd114jdfnsydb_4mj4fw0000gn/T/ipykernel_69108/1267822447.py:47: DeprecationWarning: Legacy PRNGKey detected in GeneticEngine.init_state(). For explicit PRNG backend control use malthusjax.core.random.create_key() or jax.random.key()
  init_state = engine.init_state(jr.PRNGKey(seed))


┌─ live crossover operator ──────────────────────┐
│ type: malthusjax.operators.crossover.evosax_crossover.EvosaxUniformCrossoverWrapper
│ result: {'offspring_shape': (12, 3), 'first_offspring': [[0.2873826026916504, -1.1607170104980469, -3.1087136268615723], [0.37892818450927734, -0.17350316047668457, 0.18710017204284668], [-0.7644712924957275, -1.5434801578521729, 4.293442726135254]]}
│ parameters:
│   num_offspring: 1
│   input_length: 12
│   typed_keys: False
│   max_generations: 20
│   crossover_rate: 0.1
│   injection_mode: True
└──────────────────────────────────────────────────────────┘
┌─ live mutation operator ───────────────────────┐
│ type: malthusjax.operators.mutation.evosax_mutation.EvosaxGaussianWrapper
│ result: {'offspring_shape': (12, 3), 'first_offspring': [[0.2873826026916504, -1.1607170104980469, -3.1087136268615723], [0.37892818450927734, -0.17350316047668457, 0.18710017204284668], [-0.7644712924957275, -1.5434801578521729, 4.293442726135254]]}
│ parameters:
│   

## Replay Modes

The replay helpers are useful when you have already captured a trace or `.npz` bundle and want to verify parity against the exact same RNG stream.

These modes are narrower than the full backend runs: they are intended to answer one question at a time, such as whether the sampled parent indices or crossover keys match the captured trace.

In [13]:
def replay_crossover_parity() -> None:
    trace_path = Path(f'results/crossover_trace_{seed}.npz')
    if not trace_path.exists():
        raise FileNotFoundError(f'Missing captured trace: {trace_path}')

    data = np.load(trace_path)
    config = RealGenomeConfig(shape=(dimensions,), bounds=(-5.0, 5.0))

    def _fmt(value: object) -> str:
        arr = np.asarray(value)
        if arr.ndim == 0:
            return repr(arr.item())
        preview = arr.reshape(-1)[:6].tolist()
        return f'shape={arr.shape}, preview={preview}'

    def _compare(label: str, live: object, captured: object) -> None:
        live_arr = np.asarray(live)
        captured_arr = np.asarray(captured)
        if live_arr.shape == captured_arr.shape:
            diff = np.abs(live_arr - captured_arr)
            diff_text = (
                f'max_diff={float(diff.max()):.6e}, '
                f'mean_diff={float(diff.mean()):.6e}, '
                f'allclose={bool(np.allclose(live_arr, captured_arr))}'
            )
        else:
            diff_text = f'shape_mismatch live={live_arr.shape} captured={captured_arr.shape}'
        print(f'{label}: live={_fmt(live_arr)} | captured={_fmt(captured_arr)} | {diff_text}')

    evaluator = BBOBEvaluator.create(
        BBOBConfig(
            fn_name=function,
            num_dims=dimensions,
            seed=seed,
            maximize=False,
        )
    )
    genome_config = config
    local_elite_k = _resolved_elite_k()
    engine = GeneticEngine(
        engine_params=GeneticEngineParams(
            pop_size=pop_size,
            elitism=0,
            num_generations=1,
            forward_presplit_keys=True,
        ),
        genome_config=genome_config,
        evaluator=evaluator,
        selection=ElitePoolSelection(num_selections=pop_size, elite_k=local_elite_k),
        crossover=EvosaxUniformCrossoverWrapper(
            num_offspring=1,
            crossover_rate=crossover_rate,
        ),
        mutation=EvosaxGaussianWrapper(
            num_offspring=1,
            mutation_strength=mutation_strength,
        ),
        enable_progress_bar=False,
    )

    live_state = engine.init_state(jr.PRNGKey(seed))
    live_k_sel, live_k_cross, live_k_mut, live_k_next = engine._allocate_entropy(live_state)
    live_elites, live_parent_indices = engine._selection_phase(
        live_k_sel, live_state.population, live_state.operators, engine.engine_params
    )
    live_num_pairs = live_state.resource_map.crossover.input_count // 2
    live_p1_idx = live_parent_indices[:live_num_pairs]
    live_p2_idx = live_parent_indices[live_num_pairs : live_num_pairs * 2]
    live_p1_genes = jax.tree_util.tree_map(lambda x: x[live_p1_idx], live_state.population.genes)
    live_p2_genes = jax.tree_util.tree_map(lambda x: x[live_p2_idx], live_state.population.genes)
    live_dummy = jnp.zeros(live_num_pairs)
    live_p1_pop = live_state.population.spawn_offspring(live_p1_genes, fitness=live_dummy)
    live_p2_pop = live_state.population.spawn_offspring(live_p2_genes, fitness=live_dummy)
    live_mutants = engine._reproduction_phase(
        live_k_cross,
        live_k_mut,
        live_parent_indices,
        live_state.population,
        live_state.operators,
        live_state.resource_map,
        generation=live_state.generation,
    )
    live_next_genes = engine._merge(live_elites, live_mutants.genes, live_state)
    live_new_pop = engine._evaluate(live_next_genes, live_state)

    captured_live_mj = np.asarray(data['mj_offspring'])
    captured_replay = np.asarray(data['ev_offspring'])
    captured_k_cross = np.asarray(data['k_cross_keys'])
    captured_p1_idx = np.asarray(data['p1_idx'])
    captured_p2_idx = np.asarray(data['p2_idx'])
    captured_p1 = np.asarray(data['p1_pop'])
    captured_p2 = np.asarray(data['p2_pop'])

    print('\n== live vs captured replay (side-by-side) ==')
    print(f'trace: {trace_path}')
    _compare('k_cross', live_k_cross, captured_k_cross)
    _compare('p1_idx', live_p1_idx, captured_p1_idx)
    _compare('p2_idx', live_p2_idx, captured_p2_idx)
    _compare('p1_first_leaf', jax.tree_util.tree_leaves(live_p1_pop.genes)[0], captured_p1)
    _compare('p2_first_leaf', jax.tree_util.tree_leaves(live_p2_pop.genes)[0], captured_p2)
    _compare('live_offspring_vs_captured_mj', jax.tree_util.tree_leaves(live_mutants.genes)[0], captured_live_mj)
    _compare('live_offspring_vs_captured_replay', jax.tree_util.tree_leaves(live_mutants.genes)[0], captured_replay)
    _compare('live_next_genes_vs_trace_mj', jax.tree_util.tree_leaves(live_next_genes)[0], captured_live_mj)
    _compare('live_fitness', live_new_pop.fitness, data['fitness_init'])

def replay_selection_parity() -> None:
    trace_path = Path(f'results/crossover_trace_{seed}.npz')
    if not trace_path.exists():
        raise FileNotFoundError(f'Missing captured trace: {trace_path}')

    data = np.load(trace_path)
    config = RealGenomeConfig(shape=(dimensions,), bounds=(-5.0, 5.0))
    population_init = jnp.asarray(data['population_init'])
    fitness_init = jnp.asarray(data['fitness_init'])
    evaluator = BBOBEvaluator.create(
        BBOBConfig(
            fn_name=function,
            num_dims=dimensions,
            seed=seed,
            maximize=False,
        )
    )
    local_elite_k = _resolved_elite_k()
    engine = GeneticEngine(
        engine_params=GeneticEngineParams(
            pop_size=pop_size,
            elitism=0,
            num_generations=1,
            forward_presplit_keys=True,
        ),
        genome_config=config,
        evaluator=evaluator,
        selection=ElitePoolSelection(num_selections=pop_size, elite_k=local_elite_k),
        crossover=EvosaxUniformCrossoverWrapper(num_offspring=1, crossover_rate=crossover_rate),
        mutation=EvosaxGaussianWrapper(num_offspring=1, mutation_strength=mutation_strength),
        enable_progress_bar=False,
    )

    live_state = engine.init_state(jr.PRNGKey(seed))
    evaluated_population = evaluator.evaluate_population(
        RealPopulation.from_array(np.asarray(population_init), config, axis=0)
    )
    live_state = live_state.replace(
        population=evaluated_population,
        best_genome=evaluated_population.genes[0],
        best_fitness=evaluated_population.fitness[0],
    )

    alloc_k_sel, _, _, _ = engine._allocate_entropy(live_state)
    if 'k_parent_keys' in data.files:
        try:
            parent_keys = jnp.asarray(data['k_parent_keys'])
            live_k_sel = parent_keys
        except Exception:
            base_key, parent_key = jr.split(jr.PRNGKey(seed + 3), 2)
            k1, k2 = jr.split(parent_key, 2)
            live_k_sel = jnp.stack([k1, k2])
    else:
        base_key, parent_key = jr.split(jr.PRNGKey(seed + 3), 2)
        k1, k2 = jr.split(parent_key, 2)
        live_k_sel = jnp.stack([k1, k2])

    live_elites, live_parent_indices = engine._selection_phase(
        live_k_sel, live_state.population, live_state.operators, engine.engine_params
    )

    captured_p1_idx = np.asarray(data['p1_idx'])
    captured_p2_idx = np.asarray(data['p2_idx'])
    captured_p1 = np.asarray(data['p1_pop'])
    captured_p2 = np.asarray(data['p2_pop'])
    captured_parent_indices = np.concatenate([captured_p1_idx, captured_p2_idx])
    live_parent_indices_arr = np.asarray(live_parent_indices)

    print('\n== live vs captured selection (side-by-side) ==')
    print(f'trace: {trace_path}')
    print(
        f'live_parent_indices: shape={live_parent_indices_arr.shape}, '
        f'preview={live_parent_indices_arr[:8].tolist()}'
    )
    print(
        f'captured_parent_indices: shape={captured_parent_indices.shape}, '
        f'preview={captured_parent_indices[:8].tolist()}'
    )
    print(
        'selection compare: '
        f'max_diff={float(np.abs(live_parent_indices_arr - captured_parent_indices).max()):.6e}, '
        f'mean_diff={float(np.abs(live_parent_indices_arr - captured_parent_indices).mean()):.6e}, '
        f'allclose={bool(np.array_equal(live_parent_indices_arr, captured_parent_indices))}'
    )

    live_num_pairs = live_state.resource_map.crossover.input_count // 2
    live_p1_idx = live_parent_indices_arr[:live_num_pairs]
    live_p2_idx = live_parent_indices_arr[live_num_pairs : live_num_pairs * 2]
    print(
        f'live p1_idx preview={live_p1_idx[:8].tolist()} | '
        f'captured p1_idx preview={captured_p1_idx[:8].tolist()}'
    )
    print(
        f'live p2_idx preview={live_p2_idx[:8].tolist()} | '
        f'captured p2_idx preview={captured_p2_idx[:8].tolist()}'
    )

    live_p1_genes = jax.tree_util.tree_map(lambda x: x[live_p1_idx], live_state.population.genes)
    live_p2_genes = jax.tree_util.tree_map(lambda x: x[live_p2_idx], live_state.population.genes)
    live_dummy = jnp.zeros(live_num_pairs)
    live_p1_pop = live_state.population.spawn_offspring(live_p1_genes, fitness=live_dummy)
    live_p2_pop = live_state.population.spawn_offspring(live_p2_genes, fitness=live_dummy)

    live_p1_arr = np.asarray(jax.tree_util.tree_leaves(live_p1_pop.genes)[0])
    live_p2_arr = np.asarray(jax.tree_util.tree_leaves(live_p2_pop.genes)[0])
    print(f'p1_parent_genes: live shape={live_p1_arr.shape}, captured shape={captured_p1.shape}')
    print(f'p2_parent_genes: live shape={live_p2_arr.shape}, captured shape={captured_p2.shape}')
    print(
        f"fitness_init parity check: allclose={bool(np.array_equal(np.asarray(fitness_init),
        np.asarray(data['fitness_init'])))}"
    )

## How To Use This Notebook

1. Change the configuration cell near the top.
2. Run the helper cells from top to bottom.
3. Inspect the backend reports.
4. If you have a captured trace file, set the replay mode flags and rerun the replay cells.

The detailed comparison sweeps from the script are intentionally left out of the notebook body so the walkthrough stays readable. If you want, we can add those as an appendix section in a follow-up notebook revision.